# 02 · Cube scene adaptive multiplexing (RGB)

In [ ]:
%load_ext autoreload
%autoreload 2
import mitsuba as mi
import numpy as np
from matplotlib import pyplot as plt

from utils.utils import render_scene, image_stats, unroll_image, roll_image, linear_to_srgb
from tqdm import tqdm

MI_HARDWARE_TYPE = "cuda_ad"
MI_COLOR_TYPE = "rgb"
DOUBLE_PRECISION = "" # float32
mi.set_variant(f"{MI_HARDWARE_TYPE}_{MI_COLOR_TYPE}{DOUBLE_PRECISION}")
f"{MI_HARDWARE_TYPE}_{MI_COLOR_TYPE}{DOUBLE_PRECISION}"

plt.rcParams['image.cmap'] = 'gray'


In [ ]:
# constants

LIGHT_H = 128
LIGHT_W = 128
CAM_H = 128
CAM_W = 128
SAMPLES = 4096
PLANE_SCALE = 2.5
EMITTER_RADIANCE = 100
FORWARD_AREA_LIGHT_PATTERN = np.ones([LIGHT_H, LIGHT_W, 3]) * EMITTER_RADIANCE
BACKWARD_AREA_LIGHT_PATTERN = np.ones([CAM_H, CAM_W, 3]) * EMITTER_RADIANCE
FOV = 45
THRESHOLD = 0.0005

CAMERA_LOC = [4, 8, 12]
CAMERA_TARGET = [0, 0, 0]
LIGHT_LOC = [9, 3, 5]
LIGHT_TARGET = CAMERA_TARGET
UP = [0, 1, 0]

In [ ]:
base_scene = {
    "type": "scene",
    "integrator": {"type": "path"},  # Path tracer
    "cube": {
        "type": "cube",
        "to_world": mi.ScalarTransform4f.translate([0, 0, 0]),
        "bsdf": {
            "type": "diffuse",
            "reflectance": {"type": "rgb", "value": [0.9, 0.1, 0.1]},
        },
    },
    "ground_plane": {
        "type": "cube",
        "to_world": mi.ScalarTransform4f.translate([0, -1, 0])
        @ mi.ScalarTransform4f.scale([PLANE_SCALE, 0.001, PLANE_SCALE]),
        "bsdf": {
            "type": "diffuse",
            "reflectance": {"type": "rgb", "value": [0.6, 0.6, 0.6]},
        },
    },
    "left_plane": {
        "type": "cube",
        "to_world": mi.ScalarTransform4f.translate([-PLANE_SCALE, 0, 0])
        @ mi.ScalarTransform4f.scale([0.01, PLANE_SCALE, PLANE_SCALE]),
        "bsdf": {
            "type": "diffuse",
            "reflectance": {"type": "rgb", "value": [0.3, 0.9, 0.3]},
        },
    },
    "right_plane": {
        "type": "cube",
        "to_world": mi.ScalarTransform4f.translate([0, 0, -PLANE_SCALE])
        @ mi.ScalarTransform4f.scale([PLANE_SCALE, PLANE_SCALE, 0.01]),
        "bsdf": {
            "type": "diffuse",
            "reflectance": {"type": "rgb", "value": [0.3, 0.3, 0.9]},
        },
    },
}

In [ ]:
forward_scene = {}
for key, val in base_scene.items():
    forward_scene[key] = val

forward_scene["sensor"] = {
    "type": "perspective",
    "fov": FOV,
    "to_world": mi.ScalarTransform4f.look_at(
        origin=CAMERA_LOC, target=CAMERA_TARGET, up=UP
    ),
    "film": {
        "type": "hdrfilm",
        "pixel_format": "rgb",
        "width": CAM_H,
        "height": CAM_W,
        "rfilter": {"type": "box"},
    },
    "sampler": {"type": "independent", "sample_count": SAMPLES},
}

# SET AREA LIGHT
forward_scene["textured_light"] = {
    "type": "projector",
    "to_world": mi.ScalarTransform4f.look_at(
        origin=LIGHT_LOC, target=LIGHT_TARGET, up=UP
    ),
    "fov": FOV,
    "irradiance": {
        "type": "bitmap",
        "bitmap": mi.Bitmap(FORWARD_AREA_LIGHT_PATTERN),
        "raw": True,
        "filter_type": "nearest",  # CRUCIAL to avoid extra illumination at corners
    },
}

forward_luminance = render_scene(forward_scene)
plt.imshow(forward_luminance)
plt.title("forward image")

print(image_stats(forward_luminance))

In [ ]:
backward_scene = {}
for key, val in base_scene.items():
    backward_scene[key] = val

backward_scene["sensor"] = {
    "type": "perspective",
    "to_world": mi.ScalarTransform4f.look_at(
        origin=LIGHT_LOC, target=LIGHT_TARGET, up=UP
    ),
    "fov": FOV,
    "film": {
        "type": "hdrfilm",
        "pixel_format": "rgb",
        "width": LIGHT_H,
        "height": LIGHT_W,
        "rfilter": {"type": "box"},
    },
    "sampler": {"type": "independent", "sample_count": SAMPLES},
}

# SET AREA LIGHT
backward_scene["textured_light"] = {
    "type": "projector",
    "to_world": mi.ScalarTransform4f.look_at(
        origin=CAMERA_LOC, target=CAMERA_TARGET, up=UP
    ),
    "fov": FOV,
    "irradiance": {
        "type": "bitmap",
        "bitmap": mi.Bitmap(BACKWARD_AREA_LIGHT_PATTERN),
        "raw": True,
        "filter_type": "nearest",  # CRUCIAL to avoid extra illumination at corners
    },
}

backward_luminance = render_scene(backward_scene)
plt.imshow(backward_luminance)
plt.title("backward image")
print(image_stats(backward_luminance))

In [ ]:
# SETUP AND UTILS

def get_light_bitmap_at_row_col(row: int, col: int):
    tex = np.zeros([LIGHT_H, LIGHT_W])
    tex[row][col] = EMITTER_RADIANCE
    return mi.Bitmap(tex)


def update_area_light(scene_dict: dict, bitmap: mi.Bitmap, forward: bool = True):
    origin, target = LIGHT_LOC, LIGHT_TARGET
    if not forward:
        origin, target = CAMERA_LOC, CAMERA_TARGET
    scene_dict["textured_light"] = {
        "type": "projector",
        "to_world": mi.ScalarTransform4f.look_at(origin=origin, target=target, up=UP),
        "fov": FOV,
        "irradiance": {
            "type": "bitmap",
            "bitmap": bitmap,
            "raw": True,
            "filter_type": "nearest",  # CRUCIAL to avoid extra illumination at corners
        },
    }
    return scene_dict


scene_dict = {}
for key, val in base_scene.items():
    scene_dict[key] = val

scene_dict["sensor"] = {
    "type": "perspective",
    "fov": FOV,
    "to_world": mi.ScalarTransform4f.look_at(
        origin=CAMERA_LOC, target=CAMERA_TARGET, up=UP
    ),
    "film": {
        "type": "hdrfilm",
        "pixel_format": "rgb",
        "width": CAM_H,
        "height": CAM_W,
        "rfilter": {"type": "box"},
    },
    "sampler": {"type": "independent", "sample_count": SAMPLES},
}

In [ ]:
# scan light
from sampling_algorithms.adaptive_multiplexing import AdaptiveMultiplexing

scanner = AdaptiveMultiplexing(
    camera_resolution=(CAM_H, CAM_W, 3),
    projector_resolution=(LIGHT_H, LIGHT_W)
)

total_patterns_projected = 0

while not scanner.finished:
    patterns = scanner.get_next_patterns()
    if not patterns:
        break
    
    current_round_captures = []
    total_patterns_projected += len(patterns)
    
    for pattern in tqdm(patterns, desc=f"Scanning Phase (Blocks: {len(scanner.blocks_to_process)})"):
        radiance_pattern = pattern * EMITTER_RADIANCE
        radiance_pattern = np.stack([radiance_pattern]*3, axis=-1)
        
        bitmap = mi.Bitmap(radiance_pattern)
        scene_dict = update_area_light(scene_dict, bitmap)
        
        luminance = render_scene(scene_dict)
        current_round_captures.append(luminance)
    
    scanner.process_captured_images(current_round_captures, threshold=THRESHOLD)

print(f"Total Patterns Projected: {total_patterns_projected}")

T_matrix_flat = scanner.get_T_matrix()
# Reshape assuming flat T_matrix is (CAM_PIXELS*3, PROJ_PIXELS)
# AdaptiveMultiplexing t_entries stores flattened camera pixel index including RGB
T_matrix = T_matrix_flat.reshape(CAM_H * CAM_W, 3, LIGHT_H * LIGHT_W).transpose(0, 2, 1)


In [ ]:
c = np.ones([LIGHT_H * LIGHT_W, 3])
p = np.einsum('hwc, wc -> hc', T_matrix, c)
recon = roll_image(p, CAM_H, CAM_W, 3)

print("ROLLED IMAGE STATS:\n", image_stats(recon))
print("FORWARD_LUMINANCE STATS:\n", image_stats(forward_luminance))
diff = forward_luminance - recon
print(forward_luminance.shape)
print(recon.shape)
print("DIFF STATS:", image_stats(diff))

fig, ax = plt.subplots(1, 3, figsize=(10, 5))
ax[0].imshow(recon)
ax[1].imshow(forward_luminance)
ax[2].imshow(diff)
plt.tight_layout()
plt.savefig("../results/cube/adaptive/reconstruction_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
dual_c = np.ones([CAM_H * CAM_W, 3])
T_transpose = T_matrix.swapaxes(0, 1)   # now shape = (W, H, 3)
dual_p = np.einsum('hwc, wc -> hc', T_transpose, dual_c)
plt.show()

dual_rolled = roll_image(dual_p, LIGHT_H, LIGHT_W, 3)
plt.imshow(dual_rolled)
plt.show()
plt.imshow(backward_luminance)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(forward_luminance)
ax[0].set_title("primal image")
ax[1].imshow(dual_rolled)
ax[1].set_title("dual image")
ax[2].imshow(backward_luminance)
ax[2].set_title("ground truth dual image")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.savefig("../results/cube/adaptive/dual_image_comparison.png", bbox_inches="tight")